In [1]:
import json

def read_config(filepath):
    try:
        with open(filepath,"r") as file:
            config = json.load(file)
            print("config loaded successfully:")
            print(config)
            return config 
        
    except FileNotFoundError as e:
        print("Error: file not found",e)
        
    except json.JSONDecodeError:
        print("Error:invalid json format")
        
    finally:
        print("Read attempt complete")
        
# Create a sample valid JSON file for testing
with open("config.json","w") as file:
    json.dump({"username":"admin","theme":"dark"},file)

print("valid file")
config=read_config("config.json")
print(config)

print("missing file test:")
config=read_config("missing.json")
print(config)

valid file
config loaded successfully:
{'username': 'admin', 'theme': 'dark'}
Read attempt complete
{'username': 'admin', 'theme': 'dark'}
missing file test:
Error: file not found [Errno 2] No such file or directory: 'missing.json'
Read attempt complete
None


E2
Type-safe data parser Easy
The CSV reader returns every value as a string. Write parse_row(row) to cast salary to int and join_date to a Python date. Catch ValueError for bad salary and ValueError for bad date, printing which field failed.
rows = [
    {"name": "Riya", "salary": "85000", "join_date": "2022-03-15"},
    {"name": "Sam",  "salary": "not_a_number", "join_date": "2021-07-01"},
    {"name": "Kiran","salary": "92000", "join_date": "invalid-date"}
]

In [2]:
from datetime import datetime

def parse_row(row):
    try:
        row["salary"] = int(row["salary"])
    except ValueError as v:
        print(f"salary error for {row['name']}", v)
    
    try:
        row["join_date"]=datetime.strptime(row["join_date"],"%Y-%m-%d").date()
        
    except ValueError as v:
        print(f"join date error for {row['name']}: invalid date '{row['join_date']}'",v)
    
    return row

rows = [
    {"name": "Riya", "salary": "85000", "join_date": "2022-03-15"},
    {"name": "Sam",  "salary": "not_a_number", "join_date": "2021-07-01"},
    {"name": "Kiran","salary": "92000", "join_date": "invalid-date"}
]

for row in rows:
    print(parse_row(row))


{'name': 'Riya', 'salary': 85000, 'join_date': datetime.date(2022, 3, 15)}
salary error for Sam invalid literal for int() with base 10: 'not_a_number'
{'name': 'Sam', 'salary': 'not_a_number', 'join_date': datetime.date(2021, 7, 1)}
join date error for Kiran: invalid date 'invalid-date' time data 'invalid-date' does not match format '%Y-%m-%d'
{'name': 'Kiran', 'salary': 92000, 'join_date': 'invalid-date'}


Custom exception — pipeline validation Medium
Define a custom exception PipelineConfigError. Write a function validate_pipeline(config) that raises it with a descriptive message if: retries is not between 1–5, schedule is missing, or source is empty. Test with both valid and invalid configs.
valid_config = {"source": "BigQuery", "schedule": "0 2 * * *", "retries": 3}
bad_config1  = {"source": "BigQuery", "schedule": "0 2 * * *", "retries": 10}
bad_config2  = {"source": "", "retries": 2}

also explain the que and solution

In [3]:
class PipelineConfigError(Exception):
    """Custom exception for invalid pipeline configuration"""
    pass

def validate_pipeline(config):
    if not config.get("source"):
        raise PipelineConfigError("source cannot by empty")

    if "schedule" not in config:
        raise PipelineConfigError("schedule is missing")

    retries = config.get("retries")
    if retries not in range(1,6):
        raise PipelineConfigError("retries must be between 1 and 5.")
    
    print("Pipeline configuration is valid")
        
# Test data
valid_config = {
    "source": "BigQuery",
    "schedule": "0 2 * * *",
    "retries": 3
}

bad_config1 = {
    "source": "BigQuery",
    "schedule": "0 2 * * *",
    "retries": 10
}

bad_config2 = {
    "source": "",
    "retries": 2
}

configs = [valid_config,bad_config1,bad_config2]

for i,config in enumerate(configs,start=1):
    print(f"\ntesting config {1}:")
    try:
        validate_pipeline(config)
    except PipelineConfigError as e:
        print("error:",e)


testing config 1:
Pipeline configuration is valid

testing config 1:
error: retries must be between 1 and 5.

testing config 1:
error: source cannot by empty



raise + re-raise pattern Medium
Write load_to_db(records) that raises a ValueError if records is empty. Inside a calling function run_pipeline(), catch it, log "Pipeline aborted: {reason}", then re-raise it using raise so the outer caller still sees the exception. Demonstrate the chain with a try/except in main().

In [5]:
def load_to_db(records):
    if not records:
        raise ValueError("No records to load")

    print(f"loaded {len(records)} record into the db")
    
def run_pipeline(records):
    try:
        load_to_db(records)
        print("pipeline completed successfully")
    except ValueError as e:
        print(f"Pipeline aborted: {e}")
        raise
def main():
    try:
        run_pipeline([])
    except ValueError as e:
        print(f"main caught exception: {e}")
        
if __name__ == "__main__":
    main()

Pipeline aborted: No records to load
main caught exception: No records to load


Wrap an entire ETL step Medium
Write a function safe_transform(filepath) that: reads a CSV, parses each row, skips bad rows (logging the error), and returns the clean rows. It should never crash — only log and continue. Use the employee data from C4.
This is exactly what production DE code looks like — never let one bad row kill the pipeline.

employees = [
    {"emp_id":"E01","name":"Arjun","department":"Engineering","salary":"85000"},
    {"emp_id":"E02","name":"Meera","department":"HR","salary":"60000"},
    {"emp_id":"E03","name":"Riya","department":"Engineering","salary":"65000"},
    {"emp_id":"E04","name":"Kiran","department":"Engineering","salary":"92000"},
    {"emp_id":"E05","name":"Sam","department":"Finance","salary":"71000"}
]

In [8]:
import csv


employees = [
    {"emp_id":"E01","name":"Arjun","department":"Engineering","salary":"85000"},
    {"emp_id":"E02","name":"Meera","department":"HR","salary":"60000"},
    {"emp_id":"E03","name":"Riya","department":"Engineering","salary":"65000"},
    {"emp_id":"E04","name":"Kiran","department":"Engineering","salary":"92000"},
    {"emp_id":"E05","name":"Sam","department":"Finance","salary":"71000"}
]

with open("employees.csv","w",newline="") as f:
    writer = csv.DictWriter(
        f,fieldnames=["emp_id", "name", "department", "salary"]
    )
    writer.writeheader()
    writer.writerows(employees)
    
with open("employees.csv","a",newline="") as f:
    f.write("E06,Alice,Sales,abc\n")
    f.write("E07,Bob,marketing,\n")

In [11]:
def safe_transform(filepath):
    """reads a csv parses each row,skips bad rows,logs error,and returns only clean rows"""
    
    clean_rows=[]
    
    try:
        with open(filepath,"r",newline="") as f:
            reader=csv.DictReader(f)
            
            for row_num,row in enumerate(reader,start=2): #header is row 1 
                try:
                    salary=int(row["salary"]) #parse salary
                    clean_rows.append({
                        "emp_id":row["emp_id"],
                        "name":row["name"],
                        "department":row["department"],
                        "salary":salary
                    })
                except Exception as e:
                    print(f"skipping row {row_num}: {e}")
    except Exception as e:
        print(f"error reading file {filepath}:{e} ")
        
    return clean_rows

rows=safe_transform("employees.csv")

print("\nClean rows:")
for r in rows:
    print(r)

skipping row 7: invalid literal for int() with base 10: 'abc'
skipping row 8: invalid literal for int() with base 10: ''

Clean rows:
{'emp_id': 'E01', 'name': 'Arjun', 'department': 'Engineering', 'salary': 85000}
{'emp_id': 'E02', 'name': 'Meera', 'department': 'HR', 'salary': 60000}
{'emp_id': 'E03', 'name': 'Riya', 'department': 'Engineering', 'salary': 65000}
{'emp_id': 'E04', 'name': 'Kiran', 'department': 'Engineering', 'salary': 92000}
{'emp_id': 'E05', 'name': 'Sam', 'department': 'Finance', 'salary': 71000}
